# NefroQuest — Revisão de Questões via GPT-4o

**Fluxo:**
1. Baixa `index.html`, `review_questions.py`, `apply_reviews.py` do GitHub
2. Revisa as 1034 questões em **11 lotes de 100**
3. Aplica as revisões cumulativamente no `index.html`
4. Baixa o `index.html` final para commit

**Antes de rodar:** adicione `OPENAI_API_KEY` e `GITHUB_TOKEN` nos **Secrets** do Colab (ícone de chave na barra lateral).

In [ ]:
# ─────────────────────────────────────────────
# CÉLULA 1 — Instalação de dependências
# ─────────────────────────────────────────────
!pip install -q openai
print('openai instalado.')

In [ ]:
# ─────────────────────────────────────────────
# CÉLULA 2 — Configuração: API Key + Token GitHub
# ─────────────────────────────────────────────
import os
from google.colab import userdata

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')  # Personal Access Token (repo scope)
GITHUB_REPO  = 'orlandobrunet-sketch/base-verification'
GITHUB_BRANCH = 'main'  # branch de onde baixar o index.html

print('Configuração OK.')
print(f'  OpenAI key: {os.environ["OPENAI_API_KEY"][:10]}...')
print(f'  GitHub repo: {GITHUB_REPO} / branch: {GITHUB_BRANCH}')

In [ ]:
# ─────────────────────────────────────────────
# CÉLULA 3 — Download dos arquivos do GitHub
# ─────────────────────────────────────────────
import urllib.request

FILES = ['index.html', 'review_questions.py', 'apply_reviews.py']
BASE_URL = f'https://raw.githubusercontent.com/{GITHUB_REPO}/{GITHUB_BRANCH}'

headers = {'Authorization': f'token {GITHUB_TOKEN}'} if GITHUB_TOKEN else {}

for fname in FILES:
    url = f'{BASE_URL}/{fname}'
    req = urllib.request.Request(url, headers=headers)
    with urllib.request.urlopen(req) as resp, open(fname, 'wb') as out:
        out.write(resp.read())
    size = os.path.getsize(fname)
    print(f'OK  {fname:<25} {size:>10,} bytes')

# Verifica contagem de questões
import re
with open('index.html', encoding='utf-8') as f:
    html = f.read()
n = len(re.findall(r'"t":', html[html.find('const topics = ['):html.find('\n];', html.find('const topics = ['))]))
print(f'\nQuestões encontradas no banco: {n}')

In [ ]:
# ─────────────────────────────────────────────
# CÉLULA 4 — Definição dos lotes (11 × 100)
# ─────────────────────────────────────────────
TOTAL     = 1034
BATCH_SZ  = 100

batches = []
s = 0
while s < TOTAL:
    e = min(s + BATCH_SZ, TOTAL)
    batches.append((s, e))
    s = e

print(f'{len(batches)} lotes definidos:')
for i, (s, e) in enumerate(batches):
    fname = f'reviewed_{s}_{e}.json'
    done = os.path.exists(fname)
    status = 'DONE' if done else '----'
    print(f'  Lote {i:02d}: questões {s:4d}–{e:4d}  [{status}]  → {fname}')

In [ ]:
# ─────────────────────────────────────────────
# CÉLULA 5 — Rodar UM lote específico
# (use esta célula se quiser controle manual)
# ─────────────────────────────────────────────
LOTE_START = 0    # ← altere aqui
LOTE_END   = 100  # ← altere aqui
OUT_FILE   = f'reviewed_{LOTE_START}_{LOTE_END}.json'

!python review_questions.py --start {LOTE_START} --end {LOTE_END} --out {OUT_FILE}

In [ ]:
# ─────────────────────────────────────────────
# CÉLULA 6 — Aplicar UM lote ao index.html
# ─────────────────────────────────────────────
!python apply_reviews.py {OUT_FILE}

In [ ]:
# ─────────────────────────────────────────────
# CÉLULA 7 — RODAR TODOS OS LOTES (automático)
# Pula lotes já concluídos (checkpointing)
# ─────────────────────────────────────────────
import subprocess, time, json

for i, (s, e) in enumerate(batches):
    out = f'reviewed_{s}_{e}.json'

    # ── Checkpoint: pula lotes já feitos ──
    if os.path.exists(out):
        with open(out) as f:
            data = json.load(f)
        good = sum(1 for r in data if not r.get('_error'))
        print(f'Lote {i:02d} ({s}–{e}): JÁ FEITO ({good}/{e-s} OK) — pulando')
        continue

    print(f'\n═══════════════════════════════════════')
    print(f'Lote {i:02d} — questões {s} a {e-1}')
    print(f'═══════════════════════════════════════')

    # ── Revisão ──
    r = subprocess.run(['python', 'review_questions.py',
                        '--start', str(s), '--end', str(e),
                        '--out', out], capture_output=False)

    if r.returncode != 0:
        print(f'ERRO no lote {i:02d}! Verifique e rode a Célula 5 manualmente.')
        break

    # ── Aplica ao index.html ──
    print(f'Aplicando {out} ao index.html...')
    subprocess.run(['python', 'apply_reviews.py', out])

    # Salva snapshot do index.html por segurança
    snap = f'index_after_lote{i:02d}.html'
    import shutil
    shutil.copy('index.html', snap)
    print(f'Snapshot salvo: {snap}')

print('\n✓ Todos os lotes concluídos!')

In [ ]:
# ─────────────────────────────────────────────
# CÉLULA 8 — Relatório final
# ─────────────────────────────────────────────
import json, os

total_ok  = 0
total_err = 0

for s, e in batches:
    fname = f'reviewed_{s}_{e}.json'
    if not os.path.exists(fname):
        print(f'Lote {s}–{e}: NÃO EXECUTADO')
        continue
    with open(fname) as f:
        data = json.load(f)
    ok  = sum(1 for r in data if not r.get('_error'))
    err = len(data) - ok
    total_ok  += ok
    total_err += err
    status = 'OK' if err == 0 else f'{err} ERROS'
    print(f'Lote {s:4d}–{e:4d}: {ok}/{e-s} revisadas  [{status}]')

print(f'\nTotal: {total_ok} revisadas com sucesso | {total_err} com erro')
size = os.path.getsize('index.html')
print(f'index.html final: {size:,} bytes')

In [ ]:
# ─────────────────────────────────────────────
# CÉLULA 9 — Re-rodar questões com ERRO
# (execute após a Célula 7 se houver erros)
# ─────────────────────────────────────────────
import json, os, subprocess, time

for s, e in batches:
    fname = f'reviewed_{s}_{e}.json'
    if not os.path.exists(fname):
        continue
    with open(fname) as f:
        data = json.load(f)
    errors = [r for r in data if r.get('_error')]
    if not errors:
        continue

    print(f'\nLote {s}–{e}: {len(errors)} questões com erro — retentando...')
    from openai import OpenAI
    client = OpenAI(api_key=os.environ['OPENAI_API_KEY'])

    # Re-importa função de revisão
    import importlib.util, sys
    spec = importlib.util.spec_from_file_location('rq', 'review_questions.py')
    rq   = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(rq)

    updated = {r['_idx']: r for r in data}
    all_q   = rq.extract_questions('index.html')

    for r in errors:
        idx   = r['_idx']
        q     = all_q[idx]
        topic = q.get('t', f'Q{idx}')
        print(f'  Retentando idx={idx}: {topic[:60]}...', end=' ')
        try:
            reviewed = rq.review_question(client, q)
            reviewed['t']    = q.get('t', '')
            reviewed['diff'] = q.get('diff', 'medium')
            reviewed['cat']  = q.get('cat', '')
            reviewed['_idx'] = idx
            updated[idx] = reviewed
            print('OK')
        except Exception as ex:
            print(f'AINDA COM ERRO: {ex}')
        time.sleep(2)

    # Salva versão atualizada do lote
    new_data = [updated[r['_idx']] for r in data]
    with open(fname, 'w', encoding='utf-8') as f:
        json.dump(new_data, f, ensure_ascii=False, indent=2)

    # Re-aplica
    subprocess.run(['python', 'apply_reviews.py', fname])

print('\nRetentativa concluída.')

In [ ]:
# ─────────────────────────────────────────────
# CÉLULA 10 — Download do index.html final
# ─────────────────────────────────────────────
from google.colab import files
files.download('index.html')
print('index.html baixado. Faça o commit no GitHub.')

## Após o download

No terminal local (dentro do repo):

```bash
# Substitua o index.html pelo arquivo baixado do Colab
git add index.html
git commit -m "feat: revisar 1034 questões via GPT-4o (review_questions pipeline)"
git push -u origin claude/nefroquest-question-counter-YslTs
```

## Troubleshooting

| Problema | Solução |
|---|---|
| `RateLimitError` | Aumente `SLEEP_SEC` em `review_questions.py` (linha 22) para 3.0 |
| `JSONDecodeError` da API | GPT retornou texto fora do JSON — rode a Célula 9 |
| Colab desconectou | Rode a Célula 7 novamente — lotes já feitos são pulados |
| `NOT FOUND` no apply | O campo `"t"` foi alterado pelo GPT — verifique o JSON do lote |